In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [2]:
dfs = {}

paths = [
    r'C:\Users\nick.chedzoy_predict\Documents\Python\Sports-Analytics\Module2_Moneyball_and_Beyond\Data\Week 5\MLBAM14.csv',
    r'C:\Users\nick.chedzoy_predict\Documents\Python\Sports-Analytics\Module2_Moneyball_and_Beyond\Data\Week 5\MLBAM15.csv',
    r'C:\Users\nick.chedzoy_predict\Documents\Python\Sports-Analytics\Module2_Moneyball_and_Beyond\Data\Week 5\MLBAM16.csv',
    r'C:\Users\nick.chedzoy_predict\Documents\Python\Sports-Analytics\Module2_Moneyball_and_Beyond\Data\Week 5\MLBAM17.csv'
]

df_names = [path.split('.csv')[0][-7:] + '_df' for path in paths]

In [3]:
df_map = list(zip(df_names, paths))

In [4]:
for mp in df_map:
    df = pd.read_csv(mp[1])

    dfs[mp[0]] = df

## 1. Calculate the run value of every event

This follows the lecture method: build the start and end base/out states, estimate run expectancy from `runsFuture`, and calculate `Run_Value = runsOnPlay + End_RE - Start_RE`.

In [5]:
def calculate_run_values(df):
    re = df.copy()

    for base in (1, 2, 3):
        re[f'Start{base}'] = re[f'start{base}B'].notna().astype(int)
        re[f'End{base}'] = re[f'end{base}B'].notna().astype(int)

    re['Start_State'] = (
        re[['Start1', 'Start2', 'Start3']].astype(str).agg(''.join, axis=1)
        + ' ' + re['startOuts'].astype(str)
    )
    re['End_State'] = (
        re[['End1', 'End2', 'End3']].astype(str).agg(''.join, axis=1)
        + ' ' + re['endOuts'].astype(str)
    )

    # Match the lecture: keep completed innings and events that change state or score.
    re = re[
        ((re['Start_State'] != re['End_State']) | (re['runsOnPlay'] > 0))
        & (re['outsInInning'] == 3)
    ].copy()

    start_re = (
        re.groupby('Start_State', as_index=False)['runsFuture']
        .mean()
        .rename(columns={'runsFuture': 'Start_RE'})
    )
    re = re.merge(start_re, on='Start_State', how='left')

    # After the third out, run expectancy is zero for every base configuration.
    terminal_states = pd.DataFrame({
        'Start_State': [f'{bases:03b} 3' for bases in range(8)],
        'Start_RE': 0.0,
    })
    end_re = (
        pd.concat([start_re, terminal_states], ignore_index=True)
        .rename(columns={'Start_State': 'End_State', 'Start_RE': 'End_RE'})
    )
    re = re.merge(end_re, on='End_State', how='left')
    re['Run_Value'] = re['runsOnPlay'] + re['End_RE'] - re['Start_RE']

    return re

In [6]:
run_values = {
    season: calculate_run_values(dfs[f'MLBAM{str(season)[-2:]}_df'])
    for season in range(2014, 2018)
}

pd.DataFrame({
    'season': run_values.keys(),
    'events_with_run_values': [len(df) for df in run_values.values()],
})

,season,events_with_run_values
0,2014,183779
1,2015,183717
2,2016,184152
3,2017,184751


## 2–3. Average by event and merge the four seasons

The four season columns are merged on the event name. An outer join retains every event initially, including `Sacrifice Bunt DP`, which is absent in 2017 and is removed in the next step.

In [7]:
event_values = {
    season: (
        df.groupby('event')['Run_Value']
        .mean()
        .rename(f'RV{str(season)[-2:]}')
    )
    for season, df in run_values.items()
}

event_run_values = pd.concat(event_values.values(), axis=1, join='outer').reset_index()
assert 'Sacrifice Bunt DP' in event_run_values['event'].values
event_run_values

,event,RV14,RV15,RV16,RV17
0,Batter Interference,-0.319625,-0.363838,-0.284649,-0.430019
1,Bunt Groundout,-0.194784,-0.200346,-0.218826,-0.209411
2,Bunt Lineout,-0.303810,-0.421575,-0.352295,-0.328292
3,Bunt Pop Out,-0.316440,-0.354384,-0.342802,-0.373225
4,Catcher Interference,0.380337,0.318276,0.301623,0.399070
5,Double,0.737518,0.752039,0.743467,0.779338
6,Double Play,-0.828774,-0.854665,-0.864981,-0.897164
7,Fan interference,0.633560,0.577453,0.533316,0.590743
8,Field Error,0.462976,0.485460,0.469989,0.493206
9,Fielders Choice,0.698076,0.719351,0.701447,0.764112


## 4–5. Remove Sacrifice Bunt DP and calculate correlations

In [8]:
event_run_values = (
    event_run_values[event_run_values['event'] != 'Sacrifice Bunt DP']
    .reset_index(drop=True)
)
assert event_run_values[['RV14', 'RV15', 'RV16', 'RV17']].notna().all().all()

rv_columns = ['RV14', 'RV15', 'RV16', 'RV17']
correlation_matrix = event_run_values[rv_columns].corr()
correlation_matrix

,RV14,RV15,RV16,RV17
RV14,1.000000,0.998819,0.997951,0.997153
RV15,0.998819,1.000000,0.997570,0.996412
RV16,0.997951,0.997570,1.000000,0.994964
RV17,0.997153,0.996412,0.994964,1.000000


## 6. Run Value Sum of Squares

For each event, subtract its four-season mean run value from each season's run value, square the differences, and sum them.

In [9]:
event_mean_rv = event_run_values[rv_columns].mean(axis=1)
event_run_values['Run Value Sum of Squares'] = (
    event_run_values[rv_columns].sub(event_mean_rv, axis=0).pow(2).sum(axis=1)
)

event_run_values.sort_values('Run Value Sum of Squares', ascending=False)

,event,RV14,RV15,RV16,RV17,Run Value Sum of Squares
28,Triple Play,-1.608127,-1.688951,-1.637543,-1.471223,0.025963
23,Sac Fly DP,-0.506647,-0.581165,-0.370443,-0.543715,0.025327
0,Batter Interference,-0.319625,-0.363838,-0.284649,-0.430019,0.011787
2,Bunt Lineout,-0.303810,-0.421575,-0.352295,-0.328292,0.007724
4,Catcher Interference,0.380337,0.318276,0.301623,0.399070,0.006675
7,Fan interference,0.633560,0.577453,0.533316,0.590743,0.005113
13,Grounded Into DP,-0.744986,-0.768327,-0.773318,-0.818005,0.002792
9,Fielders Choice,0.698076,0.719351,0.701447,0.764112,0.002769
6,Double Play,-0.828774,-0.854665,-0.864981,-0.897164,0.002402
26,Strikeout - DP,-0.630963,-0.650543,-0.622787,-0.682819,0.002139


## QUIZ 1

In [10]:
correlation_pairs = correlation_matrix.where(
    np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
).stack()
strongest_pair = correlation_pairs.idxmax()
highest_ss_event = event_run_values.loc[
    event_run_values['Run Value Sum of Squares'].idxmax(), 'event'
]
flyout_rv_2014 = event_run_values.loc[
    event_run_values['event'] == 'Flyout', 'RV14'
].iat[0]

print(
    f'Strongest correlation: {strongest_pair[0]} and {strongest_pair[1]} '
    f'({correlation_pairs.max():.6f})'
)
print(f'Highest sum of squares: {highest_ss_event}')
print(f'Average 2014 Flyout run value: {flyout_rv_2014:.6f}')

Strongest correlation: RV14 and RV15 (0.998819)
Highest sum of squares: Triple Play
Average 2014 Flyout run value: -0.229247


## 7–8. Aggregate and merge player run values

Run values are summed by the `batterId` and `batterName` combination for each season. The inner join retains only player ID/name combinations with run values in all four seasons.

In [11]:
player_values = {
    season: (
        df.groupby(['batterId', 'batterName'])['Run_Value']
        .sum()
        .rename(f'RV{str(season)[-2:]}')
    )
    for season, df in run_values.items()
}

# An inner join on batterId and batterName keeps only players represented identically in every season.
common_player_values = pd.concat(player_values.values(), axis=1, join='inner').reset_index()
player_run_values = (
    common_player_values[['batterName', 'RV14', 'RV15', 'RV16', 'RV17']]
    .sort_values('batterName')
    .reset_index(drop=True)
)

assert player_run_values.notna().all().all()
assert len(player_run_values) == len(common_player_values)
assert len(player_run_values) == 372
player_run_values

,batterName,RV14,RV15,RV16,RV17
0,Abreu,46.901143,38.344478,13.395343,31.392604
1,Adames,-2.678984,-0.479537,-12.866394,-2.872021
2,Adrianza,-5.692656,-2.441872,-3.300578,0.425352
3,Aguilar,-3.060468,0.863629,-1.270121,12.154386
4,Ahmed,-5.739550,-8.505689,-23.000581,-4.176089
...,...,...,...,...,...
367,Zobrist,5.693140,22.579925,18.888483,-13.718812
368,Zunino,-9.817323,-32.407376,8.816774,4.710472
369,"d'Arnaud, T",2.065065,8.105477,-12.314196,-2.301919
370,deGrom,-5.525731,-6.258311,-8.532559,-9.500759


## 9. Player run value correlation matrix

Because the requested follow-up asks for the correlation in **player** run values between 2014 and 2016, this matrix uses the merged player-level run values.

In [12]:
player_correlation_matrix = player_run_values[rv_columns].corr()
player_correlation_matrix

,RV14,RV15,RV16,RV17
RV14,1.000000,0.466299,0.426629,0.322764
RV15,0.466299,1.000000,0.546136,0.510132
RV16,0.426629,0.546136,1.000000,0.457391
RV17,0.322764,0.510132,0.457391,1.000000


## 10. Regress 2017 player run values on 2014–2016 values

In [13]:
import statsmodels.formula.api as smf

player_rv_model = smf.ols(
    formula='RV17 ~ RV14 + RV15 + RV16',
    data=player_run_values,
).fit()
player_rv_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                   RV17   R-squared:                       0.308
Model:                            OLS   Adj. R-squared:                  0.302
Method:                 Least Squares   F-statistic:                     54.61
Date:                Tue, 17 Nov 2026   Prob (F-statistic):           3.20e-29
Time:                        15:22:59   Log-Likelihood:                -1458.4
No. Observations:                 372   AIC:                             2925.
Df Residuals:                     368   BIC:                             2941.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.0472      0.650      0.073      0.942      -1.231       1.326
RV14           0.0620      0.056      1.101      0.272      -0.049       0.173
RV15           0.3509      0.054      6.480      0.000       0.244       0.457
RV16           0.2673      0.059      4.532      0.000       0.151       0.383
==============================================================================
Omnibus:                       39.399   Durbin-Watson:                   2.152
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               89.833
Skew:                           0.548   Prob(JB):                     3.11e-20
Kurtosis:                       5.144   Cond. No.                         20.1
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

## QUIZ 2

In [14]:
player_corr_2014_2016 = player_correlation_matrix.loc['RV14', 'RV16']
regression_r_squared = player_rv_model.rsquared
rv15_coefficient = player_rv_model.params['RV15']

print(f'2014-2016 player run value correlation: {player_corr_2014_2016:.6f}')
print(f'Regression R-squared: {regression_r_squared:.6f}')
print(f'RV15 regression coefficient: {rv15_coefficient:.6f}')

2014-2016 player run value correlation: 0.426629
Regression R-squared: 0.308066
RV15 regression coefficient: 0.350930


## 11–13. Create the batting team and aggregate team run values

The batting team is the away team in the top half of an inning and the home team in the bottom half. Team run values are the sum of all event run values, and the four seasons are inner-joined by team name.

In [15]:
for df in run_values.values():
    df['team'] = np.where(df['half'] == 'top', df['away_team'], df['home_team'])

team_values = {
    season: (
        df.groupby('team')['Run_Value']
        .sum()
        .rename(f'RV{str(season)[-2:]}')
    )
    for season, df in run_values.items()
}

team_run_values = (
    pd.concat(team_values.values(), axis=1, join='inner')
    .reset_index()
    .sort_values('team')
    .reset_index(drop=True)
)

assert team_run_values.notna().all().all()
assert len(team_run_values) == 30
team_run_values

,team,RV14,RV15,RV16,RV17
0,ana,104.641532,-25.056067,-3.184209,-47.579782
1,ari,-48.253012,31.924551,15.870968,62.616716
2,atl,-81.885752,-108.464134,-84.633901,-28.259905
3,bal,46.586965,29.882644,29.297940,-23.678031
4,bos,-37.383914,65.372953,160.812546,9.445347
5,cha,-1.307578,-79.136738,-41.643638,-47.416033
6,chn,-54.987532,-3.627047,85.327151,71.002092
7,cin,-57.227567,-61.401274,-17.643638,5.420218
8,cle,-3.725729,-4.178644,48.815791,73.780465
9,col,92.456063,47.127799,122.825528,78.616716


## 14. Team run value correlation matrix

In [16]:
team_correlation_matrix = team_run_values[rv_columns].corr()
team_correlation_matrix

,RV14,RV15,RV16,RV17
RV14,1.000000,0.363681,0.261814,0.065225
RV15,0.363681,1.000000,0.437610,0.193061
RV16,0.261814,0.437610,1.000000,0.351708
RV17,0.065225,0.193061,0.351708,1.000000


## 15. Regress 2017 team run values on 2014–2016 values

In [17]:
team_rv_model = smf.ols(
    formula='RV17 ~ RV14 + RV15 + RV16',
    data=team_run_values,
).fit()
team_rv_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                   RV17   R-squared:                       0.127
Model:                            OLS   Adj. R-squared:                  0.027
Method:                 Least Squares   F-statistic:                     1.265
Date:                Tue, 17 Nov 2026   Prob (F-statistic):              0.307
Time:                        15:22:59   Log-Likelihood:                -166.93
No. Observations:                  30   AIC:                             341.9
Df Residuals:                      26   BIC:                             347.5
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.4437     12.406     -0.036      0.972     -25.944      25.056
RV14          -0.0553      0.241     -0.230      0.820      -0.550       0.440
RV15           0.0706      0.240      0.294      0.771      -0.424       0.565
RV16           0.3788      0.231      1.638      0.113      -0.097       0.854
==============================================================================
Omnibus:                        0.584   Durbin-Watson:                   1.712
Prob(Omnibus):                  0.747   Jarque-Bera (JB):                0.061
Skew:                           0.065   Prob(JB):                        0.970
Kurtosis:                       3.179   Cond. No.                         77.2
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

## QUIZ 3

In [18]:
team_corr_2014_2017 = team_correlation_matrix.loc['RV14', 'RV17']
team_regression_r_squared = team_rv_model.rsquared
team_rv16_coefficient = team_rv_model.params['RV16']

print(f'2014-2017 team run value correlation: {team_corr_2014_2017:.6f}')
print(f'Team regression R-squared: {team_regression_r_squared:.6f}')
print(f'RV16 regression coefficient: {team_rv16_coefficient:.6f}')

2014-2017 team run value correlation: 0.065225
Team regression R-squared: 0.127367
RV16 regression coefficient: 0.378831
